# SeisMambaKAN — Colab

Production notebook. Three cells:

1. **Bootstrap**: mount Drive, clone GitHub repo, change cwd.
2. **Setup**: install deps + sync processed data from Drive.
3. **Run**: train / eval / infer.

Open this file directly via:
`https://colab.research.google.com/github/huseyinokanozturk/SeisMambaKAN/blob/main/notebooks/Colab.ipynb`

## 1) Bootstrap — Drive + GitHub clone

In [ ]:
# Mount Drive and clone (or pull) the GitHub repo.
import os, subprocess
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/SeisMambaKAN'
REPO_URL = 'https://github.com/huseyinokanozturk/SeisMambaKAN.git'

if Path(REPO_DIR, '.git').exists():
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
import sys; sys.path.insert(0, REPO_DIR)
print('cwd =', os.getcwd())

## 2) Setup — install deps + sync data

`--data-mode all` copies the full dataset (≈ few GB); use `--data-mode sample` for smoke tests.

In [ ]:
# Typer is needed before run.py can launch its own commands.
!pip install -q typer rich pyyaml tqdm
!python run.py setup --data-mode sample

## 3) Train / Eval / Infer

Pick one. All flags override values from `configs/config.yaml`.
Checkpoints are mirrored to Drive automatically (see `paths.yaml -> experiments.drive_root_dir`).

In [ ]:
# Project state at a glance
!python run.py status

In [ ]:
# Train (override anything from config.yaml on the CLI) for testing: --epochs 2 --batch-size 32 --data-mode sample
!python run.py train --epochs 20 --batch-size 32 --data-mode sample

In [ ]:
# Evaluate the latest experiment on val (auto-picks exp_NNN)
!python run.py eval --split val

In [ ]:
# Plot inference on a single trace (latest exp, random index)
!python run.py infer --split test

In [ ]:
# TensorBoard
%load_ext tensorboard
%tensorboard --logdir experiments --port 6006

Temp Commands Below

In [20]:
!cd /content/SeisMambaKAN && git pull --rebase

# Patience'i 3'e indir, 30 epoch çalıştır — modelin durması lazım
!sed -i 's/patience: 12/patience: 3/' configs/config.yaml
!python run.py train --epochs 8 --batch-size 32

# Patience'i geri al
!cd /content/SeisMambaKAN && git checkout configs/config.yaml


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 3.92 KiB | 1003.00 KiB/s, done.
From https://github.com/huseyinokanozturk/SeisMambaKAN
   6e35f83..10f9fb5  main       -> origin/main
Updating 6e35f83..10f9fb5
Fast-forward
 src/trainer.py | 265 ++++++++++++++++++++++++++++++++++++++++++---------------
 1 file changed, 198 insertions(+), 67 deletions(-)
2026-05-15 18:14:44.633660: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[Trainer] applied main-config overrides: {'training.epochs': 8, 'training.batch_size': 32}
[Dataset:train] capping num_workers 12 -> 1 (shards=1,

In [ ]:
# Eval — threshold düşük
!sed -i 's/trace_threshold: 0.8/trace_threshold: 0.3/' configs/config.yaml
!sed -i 's/timestep_threshold: 0.7/timestep_threshold: 0.2/' configs/config.yaml
!sed -i 's/p_amp_threshold: 0.25/p_amp_threshold: 0.15/' configs/config.yaml
!sed -i 's/s_amp_threshold: 0.20/s_amp_threshold: 0.15/' configs/config.yaml
!python run.py eval --split val
!cd /content/SeisMambaKAN && git checkout configs/config.yaml
